# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described by a Croissant schema and distributed via a public URL. See below for how to load the metadata and perform typical exploratory data analysis tasks.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset schema and metadata from the Croissant URL using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata for basic identification
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published on: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review the dataset's record sets and their schema, referencing entities by their `@id` as defined in the Croissant metadata.

In [ ]:
# List all record sets by @id
print("Available Record Sets (by @id):")
for record_set in dataset.metadata.record_sets:
    print(f"- {record_set['@id']} : {record_set.get('name', record_set['@id'])}")

# For demonstration, we will print field details of each record set
record_set_fields = {}
for record_set in dataset.metadata.record_sets:
    rs_id = record_set['@id']
    field_ids = [field['@id'] for field in record_set.get('fields', [])]
    record_set_fields[rs_id] = field_ids
    print(f"Record Set {rs_id} fields:")
    for field in record_set.get('fields', []):
        print(f"  - {field['@id']}: {field.get('name', '')} [dataType: {field.get('dataType', '')}]")

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames. Use the `@id` for all references.

**Note:** For this source dataset, typical record set IDs might include:
- `'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'` — This is frequently the principal table for normalized protein quantifications.

Below we show how to extract data for each record set found.

In [ ]:
# Gather all available record set @ids
record_set_ids = [r['@id'] for r in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nExtracting data from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in {record_set_id}:\n", df.columns.tolist())
    print(df.head(2))

# Choose the main record set for analysis
# Use the first record set as default for the demo, or specify if known
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    print(f"\nUsing record set: {main_rs_id} for detailed analysis.")
    main_df = dataframes[main_rs_id]

## 4. Exploratory Data Analysis (EDA)
We'll process the data by filtering, normalizing, and grouping values using field `@id` references.

You'll need to select a numeric field and (optionally) a grouping field. In mass spectrometry datasets, candidates might include normalized abundance, MW (molecular weight), or coverage percentage. Adjust the field id below as needed.

In [ ]:
# Fields by @id can be determined from previous overview section
numeric_field = None
group_field = None

# Suggest likely numeric and group fields by scanning DataFrame columns
if main_rs_id is not None:
    df = dataframes[main_rs_id]
    numeric_candidates = [col for col in df.columns if df[col].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[col])]
    group_candidates = [col for col in df.columns if df[col].dtype == object and not pd.api.types.is_numeric_dtype(df[col])]
    print("Numeric candidate fields:", numeric_candidates)
    print("Group candidate fields:", group_candidates)
    # For this kind of dataset, let's try to pick one
    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]
    if len(group_candidates) > 0:
        group_field = group_candidates[0]
    
    threshold = df[numeric_field].quantile(0.75) if numeric_field else None
    print(f"\nFiltering on: {numeric_field} > {threshold}")
    filtered_df = df[df[numeric_field] > threshold] if numeric_field else df.copy()
    print(filtered_df.head())

    # Normalize
    if numeric_field:
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another field, if appropriate
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (showing mean):\n", grouped_df.head())

## 5. Visualization
Display basic visualizations such as distributions and relationships between the normalized numeric field and a group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(main_df[numeric_field], kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    if group_field and group_field in main_df.columns:
        plt.figure(figsize=(10, 6))
        # Only plot if a reasonable number of groups exist
        counts = main_df[group_field].value_counts()
        top_groups = counts.head(10).index
        subset = main_df[main_df[group_field].isin(top_groups)]
        sns.boxplot(
            x=group_field,
            y=numeric_field,
            data=subset
        )
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field} by {group_field} (top 10)')
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, overview, and explore the FAIR^2 dataset's mass spectrometry data using the `mlcroissant` library. We referenced all fields, record sets, and extraction actions using the entity `@id` as required by the Croissant schema. Continue your exploration by selecting additional fields, record sets, or by applying domain-specific transformations and visualizations.